# ⚡ Electricity & Gas Usage Analysis

Analysis of electricity (kWh) and gas (MJ) usage data from **EnergyAustralia** and **GloBird** bills.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timedelta
from collections import defaultdict
import ipywidgets as widgets
import calendar
import warnings
import sys
from pathlib import Path

# Add project root directory to sys.path
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
from src.config import OUTPUT_DIR, CONSOLIDATED_CSV_PATH

output_dir = OUTPUT_DIR
csv_path = CONSOLIDATED_CSV_PATH # Full absolute path to consolidated.csv

warnings.filterwarnings('ignore')
# ── Style setup ──
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d29',
    'axes.edgecolor': '#2d3250',
    'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0',
    'xtick.color': '#b0b0b0',
    'ytick.color': '#b0b0b0',
    'grid.color': '#2d3250',
    'grid.alpha': 0.5,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'figure.titlesize': 20,
    'figure.titleweight': 'bold',
})

# ── Colour palette ──
ELEC_COLOR = '#00d4ff'     # cyan
GAS_COLOR  = '#ff6b35'     # orange
ELEC_LIGHT = '#7aeaff'
GAS_LIGHT  = '#ffad85'
ACCENT     = '#a855f7'     # purple

print('✅ Setup complete')

In [ ]:
# ── Load and clean data ──
df = pd.read_csv(csv_path)

# Parse dates – handle both EA format (29Apr2024) and GloBird format (26-May-2025)
def parse_date(s):
    if pd.isna(s) or str(s).strip() == '':
        return pd.NaT
    s = str(s).strip()
    for fmt in ['%d-%b-%Y', '%d%b%Y', '%d %b %Y']:
        try:
            return pd.to_datetime(s, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df['Period Start'] = df['Period Start'].apply(parse_date)
df['Period End']   = df['Period End'].apply(parse_date)

# Remove duplicate (NOR.pdf == NOR (1).pdf)
df = df.drop_duplicates(subset=['Utility', 'Period Start', 'Period End', 'Total ($)'], keep='first')

# Ensure numeric columns
df['Total Usage'] = pd.to_numeric(df['Total Usage'], errors='coerce')
df['Total ($)']   = pd.to_numeric(df['Total ($)'], errors='coerce')
df['Days']        = pd.to_numeric(df['Days'], errors='coerce')

df_elec = df[df['Utility'] == 'Electricity'].copy().sort_values('Period Start').reset_index(drop=True)
df_gas  = df[df['Utility'] == 'Gas'].copy().sort_values('Period Start').reset_index(drop=True)

print(f'Loaded {len(df_elec)} electricity and {len(df_gas)} gas records')
print(f'Date range: {df["Period Start"].min().strftime("%b %Y")} → {df["Period End"].max().strftime("%b %Y")}')


In [ ]:
# ── Prorate bills into calendar months ──
# EA bills are quarterly (~92 days). We distribute usage and cost
# proportionally across each calendar month the bill covers.

def prorate_to_months(row):
    """Distribute a bill's usage and cost across the calendar months it spans."""
    start = row['Period Start']
    end   = row['Period End']
    total_days  = row['Days']
    total_usage = row['Total Usage']
    total_cost  = row['Total ($)']
    
    if pd.isna(start) or pd.isna(end) or total_days == 0:
        return []
    
    results = []
    current = start
    while current <= end:
        # End of current month
        month_end = current.replace(day=calendar.monthrange(current.year, current.month)[1])
        period_end_in_month = min(month_end, end)
        days_in_this_month = (period_end_in_month - current).days + 1
        
        fraction = days_in_this_month / total_days
        results.append({
            'year': current.year,
            'month': current.month,
            'month_label': current.strftime('%b %Y'),
            'usage': total_usage * fraction,
            'cost': total_cost * fraction,
            'days': days_in_this_month,
            'provider': row['Provider'],
        })
        
        # Move to first day of next month
        if month_end.month == 12:
            current = month_end.replace(year=month_end.year + 1, month=1, day=1)
        else:
            current = month_end.replace(month=month_end.month + 1, day=1)
    
    return results


def build_monthly(df_util):
    """Build monthly aggregated DataFrame from bill records."""
    monthly_rows = []
    for _, row in df_util.iterrows():
        monthly_rows.extend(prorate_to_months(row))
    
    mdf = pd.DataFrame(monthly_rows)
    # Aggregate by month (in case multiple bills overlap a month)
    mdf = mdf.groupby(['year', 'month', 'month_label']).agg(
        usage=('usage', 'sum'),
        cost=('cost', 'sum'),
        days=('days', 'sum'),
    ).reset_index()
    mdf = mdf.sort_values(['year', 'month']).reset_index(drop=True)
    return mdf


elec_monthly = build_monthly(df_elec)
gas_monthly  = build_monthly(df_gas)

print(f'Electricity: {len(elec_monthly)} monthly buckets')
print(f'Gas:         {len(gas_monthly)} monthly buckets')

---
## 1. Monthly Usage (Bar Chart)
Electricity usage in **kWh** and Gas usage in **MJ** — side by side with dual Y-axes.

In [ ]:
# ── Chart 1: Monthly Usage (Interactive) ──

# Merge electricity and gas monthly on (year, month)
usage_merged = pd.merge(
    elec_monthly[['year', 'month', 'month_label', 'usage']].rename(columns={'usage': 'elec_kwh'}),
    gas_monthly[['year', 'month', 'month_label', 'usage']].rename(columns={'usage': 'gas_mj'}),
    on=['year', 'month', 'month_label'],
    how='outer'
).sort_values(['year', 'month']).reset_index(drop=True)
usage_merged = usage_merged.fillna(0)

month_options = usage_merged['month_label'].tolist()

def plot_monthly_usage(utility, date_range):
    start_month, end_month = date_range
    start_idx = month_options.index(start_month)
    end_idx = month_options.index(end_month)
    
    df_filtered = usage_merged.iloc[start_idx:end_idx + 1].copy().reset_index(drop=True)
    
    fig, ax1 = plt.subplots(figsize=(16, 7))
    x = np.arange(len(df_filtered))
    
    if utility == 'Both':
        bar_width = 0.38
        bars1 = ax1.bar(x - bar_width/2, df_filtered['elec_kwh'], bar_width,
                        color=ELEC_COLOR, alpha=0.85, label='Electricity (kWh)',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        ax1.set_ylabel('Electricity Usage (kWh)', color=ELEC_COLOR, fontsize=13, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor=ELEC_COLOR)

        ax2 = ax1.twinx()
        bars2 = ax2.bar(x + bar_width/2, df_filtered['gas_mj'], bar_width,
                        color=GAS_COLOR, alpha=0.85, label='Gas (MJ)',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        ax2.set_ylabel('Gas Usage (MJ)', color=GAS_COLOR, fontsize=13, fontweight='bold')
        ax2.tick_params(axis='y', labelcolor=GAS_COLOR)

        for bar in bars1:
            h = bar.get_height()
            if h > 0:
                ax1.text(bar.get_x() + bar.get_width()/2, h + 5, f'{h:.0f}',
                         ha='center', va='bottom', fontsize=7.5, color=ELEC_LIGHT, fontweight='bold')

        for bar in bars2:
            h = bar.get_height()
            if h > 0:
                ax2.text(bar.get_x() + bar.get_width()/2, h + 50, f'{h:.0f}',
                         ha='center', va='bottom', fontsize=7.5, color=GAS_LIGHT, fontweight='bold')

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left',
                   facecolor='#1a1d29', edgecolor='#2d3250', fontsize=10)
        title_utility = "Electricity + Gas"

    elif utility == 'Electricity':
        bar_width = 0.55
        bars1 = ax1.bar(x, df_filtered['elec_kwh'], bar_width,
                        color=ELEC_COLOR, alpha=0.85, label='Electricity (kWh)',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        ax1.set_ylabel('Electricity Usage (kWh)', color=ELEC_COLOR, fontsize=13, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor=ELEC_COLOR)

        for bar in bars1:
            h = bar.get_height()
            if h > 0:
                ax1.text(bar.get_x() + bar.get_width()/2, h + 5, f'{h:.0f}',
                         ha='center', va='bottom', fontsize=8, color=ELEC_LIGHT, fontweight='bold')

        ax1.legend(loc='upper left', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=10)
        title_utility = "Electricity"

    else:  # Gas
        bar_width = 0.55
        bars2 = ax1.bar(x, df_filtered['gas_mj'], bar_width,
                        color=GAS_COLOR, alpha=0.85, label='Gas (MJ)',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        ax1.set_ylabel('Gas Usage (MJ)', color=GAS_COLOR, fontsize=13, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor=GAS_COLOR)

        for bar in bars2:
            h = bar.get_height()
            if h > 0:
                ax1.text(bar.get_x() + bar.get_width()/2, h + 50, f'{h:.0f}',
                         ha='center', va='bottom', fontsize=8, color=GAS_LIGHT, fontweight='bold')

        ax1.legend(loc='upper left', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=10)
        title_utility = "Gas"

    ax1.set_xticks(x)
    ax1.set_xticklabels(df_filtered['month_label'], rotation=45, ha='right', fontsize=9)
    ax1.set_axisbelow(True)
    ax1.grid(axis='y', linestyle='--', alpha=0.3)
    ax1.set_title(f'Monthly Energy Usage ({title_utility}: {start_month} – {end_month})', pad=20)

    fig.tight_layout()
    fig.savefig(output_dir / 'chart_monthly_usage.png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()

# Interactive controls
usage_utility_dd = widgets.Dropdown(
    options=['Both', 'Electricity', 'Gas'],
    value='Both',
    description='Utility:',
    layout=widgets.Layout(width='250px')
)

usage_date_slider = widgets.SelectionRangeSlider(
    options=month_options,
    index=(0, len(month_options)-1),
    description='Date Range:',
    layout=widgets.Layout(width='500px')
)

widgets.interact(plot_monthly_usage, utility=usage_utility_dd, date_range=usage_date_slider)


---
## 2. Average Daily Consumption per Month
Average daily usage in **kWh/day** (electricity) and **MJ/day** (gas) — side by side with dual Y-axes.

In [ ]:
# ── Chart 2: Average Daily Consumption per Month (Interactive) ──

elec_monthly['daily_kwh'] = elec_monthly['usage'] / elec_monthly['days']
gas_monthly['daily_mj']   = gas_monthly['usage'] / gas_monthly['days']

daily_merged = pd.merge(
    elec_monthly[['year', 'month', 'month_label', 'daily_kwh']],
    gas_monthly[['year', 'month', 'month_label', 'daily_mj']],
    on=['year', 'month', 'month_label'],
    how='outer'
).sort_values(['year', 'month']).reset_index(drop=True).fillna(0)

daily_month_options = daily_merged['month_label'].tolist()

def plot_daily_consumption(utility, date_range):
    start_month, end_month = date_range
    start_idx = daily_month_options.index(start_month)
    end_idx = daily_month_options.index(end_month)
    
    df_filtered = daily_merged.iloc[start_idx:end_idx + 1].copy().reset_index(drop=True)
    
    fig, ax1 = plt.subplots(figsize=(16, 7))
    x = np.arange(len(df_filtered))
    
    if utility == 'Both':
        bar_width = 0.38
        bars1 = ax1.bar(x - bar_width/2, df_filtered['daily_kwh'], bar_width,
                        color=ELEC_COLOR, alpha=0.85, label='Electricity (kWh/day)',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        ax1.set_ylabel('Electricity Usage (kWh/day)', color=ELEC_COLOR, fontsize=13, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor=ELEC_COLOR)

        ax2 = ax1.twinx()
        bars2 = ax2.bar(x + bar_width/2, df_filtered['daily_mj'], bar_width,
                        color=GAS_COLOR, alpha=0.85, label='Gas (MJ/day)',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        ax2.set_ylabel('Gas Usage (MJ/day)', color=GAS_COLOR, fontsize=13, fontweight='bold')
        ax2.tick_params(axis='y', labelcolor=GAS_COLOR)

        for bar in bars1:
            h = bar.get_height()
            if h > 0:
                ax1.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}',
                         ha='center', va='bottom', fontsize=7.5, color=ELEC_LIGHT, fontweight='bold')

        for bar in bars2:
            h = bar.get_height()
            if h > 0:
                ax2.text(bar.get_x() + bar.get_width()/2, h + 2, f'{h:.0f}',
                         ha='center', va='bottom', fontsize=7.5, color=GAS_LIGHT, fontweight='bold')

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left',
                   facecolor='#1a1d29', edgecolor='#2d3250', fontsize=10)
        title_utility = "Electricity + Gas"

    elif utility == 'Electricity':
        bar_width = 0.55
        bars1 = ax1.bar(x, df_filtered['daily_kwh'], bar_width,
                        color=ELEC_COLOR, alpha=0.85, label='Electricity (kWh/day)',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        ax1.set_ylabel('Electricity Usage (kWh/day)', color=ELEC_COLOR, fontsize=13, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor=ELEC_COLOR)

        for bar in bars1:
            h = bar.get_height()
            if h > 0:
                ax1.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}',
                         ha='center', va='bottom', fontsize=8, color=ELEC_LIGHT, fontweight='bold')

        ax1.legend(loc='upper left', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=10)
        title_utility = "Electricity"

    else:  # Gas
        bar_width = 0.55
        bars2 = ax1.bar(x, df_filtered['daily_mj'], bar_width,
                        color=GAS_COLOR, alpha=0.85, label='Gas (MJ/day)',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        ax1.set_ylabel('Gas Usage (MJ/day)', color=GAS_COLOR, fontsize=13, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor=GAS_COLOR)

        for bar in bars2:
            h = bar.get_height()
            if h > 0:
                ax1.text(bar.get_x() + bar.get_width()/2, h + 2, f'{h:.0f}',
                         ha='center', va='bottom', fontsize=8, color=GAS_LIGHT, fontweight='bold')

        ax1.legend(loc='upper left', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=10)
        title_utility = "Gas"

    ax1.set_xticks(x)
    ax1.set_xticklabels(df_filtered['month_label'], rotation=45, ha='right', fontsize=9)
    ax1.set_axisbelow(True)
    ax1.grid(axis='y', linestyle='--', alpha=0.3)
    ax1.set_title(f'Average Daily Energy Consumption ({title_utility}: {start_month} – {end_month})', pad=20)

    fig.tight_layout()
    fig.savefig(output_dir / 'chart_daily_consumption.png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()

# Interactive controls
daily_utility_dd = widgets.Dropdown(
    options=['Both', 'Electricity', 'Gas'],
    value='Both',
    description='Utility:',
    layout=widgets.Layout(width='250px')
)

daily_date_slider = widgets.SelectionRangeSlider(
    options=daily_month_options,
    index=(0, len(daily_month_options)-1),
    description='Date Range:',
    layout=widgets.Layout(width='500px')
)

widgets.interact(plot_daily_consumption, utility=daily_utility_dd, date_range=daily_date_slider)


In [ ]:
# ── Interactive Plotly Chart: Average Daily Consumption ──
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_plotly = make_subplots(specs=[[{"secondary_y": True}]])

fig_plotly.add_trace(
    go.Bar(
        x=daily_merged['month_label'],
        y=daily_merged['daily_kwh'],
        name="Electricity (kWh/day)",
        marker_color=ELEC_COLOR,
        text=[f'{val:.1f}' if val > 0 else '' for val in daily_merged['daily_kwh']],
        textposition='outside',
        hovertemplate='%{x}<br>Electricity: %{y:.2f} kWh/day<extra></extra>'
    ),
    secondary_y=False,
)

fig_plotly.add_trace(
    go.Bar(
        x=daily_merged['month_label'],
        y=daily_merged['daily_mj'],
        name="Gas (MJ/day)",
        marker_color=GAS_COLOR,
        text=[f'{val:.0f}' if val > 0 else '' for val in daily_merged['daily_mj']],
        textposition='outside',
        hovertemplate='%{x}<br>Gas: %{y:.1f} MJ/day<extra></extra>'
    ),
    secondary_y=True,
)

fig_plotly.update_layout(
    title=dict(text="<b>Average Daily Energy Consumption per Month</b>", font=dict(size=18, color='#e0e0e0')),
    paper_bgcolor='#0f1117',
    plot_bgcolor='#1a1d29',
    font=dict(color='#e0e0e0'),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(26,29,41,0.8)', bordercolor='#2d3250', borderwidth=1),
    barmode='group',
    margin=dict(l=40, r=40, t=60, b=80),
)

fig_plotly.update_xaxes(title_text="Month", showgrid=True, gridcolor='#2d3250', tickangle=-45)
fig_plotly.update_yaxes(title_text="Electricity (kWh/day)", title_font=dict(color=ELEC_COLOR), tickfont=dict(color=ELEC_COLOR), showgrid=True, gridcolor='#2d3250', secondary_y=False)
fig_plotly.update_yaxes(title_text="Gas (MJ/day)", title_font=dict(color=GAS_COLOR), tickfont=dict(color=GAS_COLOR), showgrid=False, secondary_y=True)

fig_plotly.show()


---
## 3. Monthly Bill Amount
Total monthly bill in **AUD ($)** for electricity and gas — stacked bars to show combined spend.

In [ ]:
# ── Chart 2: Monthly Bill Amount (Interactive) ──

bill_merged = pd.merge(
    elec_monthly[['year', 'month', 'month_label', 'cost']].rename(columns={'cost': 'elec_cost'}),
    gas_monthly[['year', 'month', 'month_label', 'cost']].rename(columns={'cost': 'gas_cost'}),
    on=['year', 'month', 'month_label'],
    how='outer'
).sort_values(['year', 'month']).reset_index(drop=True)
bill_merged = bill_merged.fillna(0)
bill_merged['total_cost'] = bill_merged['elec_cost'] + bill_merged['gas_cost']

bill_month_options = bill_merged['month_label'].tolist()

def plot_monthly_bill(utility, date_range):
    start_month, end_month = date_range
    start_idx = bill_month_options.index(start_month)
    end_idx = bill_month_options.index(end_month)
    
    df_filtered = bill_merged.iloc[start_idx:end_idx + 1].copy().reset_index(drop=True)
    
    fig, ax = plt.subplots(figsize=(16, 7))
    x = np.arange(len(df_filtered))
    
    if utility == 'Both':
        bars_e = ax.bar(x, df_filtered['elec_cost'], 0.65,
                        color=ELEC_COLOR, alpha=0.85, label='Electricity',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        bars_g = ax.bar(x, df_filtered['gas_cost'], 0.65,
                        bottom=df_filtered['elec_cost'],
                        color=GAS_COLOR, alpha=0.85, label='Gas',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)

        for i, total in enumerate(df_filtered['total_cost']):
            if total > 0:
                ax.text(i, total + 3, f'${total:.0f}',
                        ha='center', va='bottom', fontsize=8, color='#e0e0e0', fontweight='bold')

        avg_cost = df_filtered['total_cost'].mean()
        title_utility = "Electricity + Gas"
        ax.legend(loc='upper right', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=10)

    elif utility == 'Electricity':
        bars_e = ax.bar(x, df_filtered['elec_cost'], 0.65,
                        color=ELEC_COLOR, alpha=0.85, label='Electricity Cost',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)

        for i, cost in enumerate(df_filtered['elec_cost']):
            if cost > 0:
                ax.text(i, cost + 2, f'${cost:.0f}',
                        ha='center', va='bottom', fontsize=8, color=ELEC_LIGHT, fontweight='bold')

        avg_cost = df_filtered['elec_cost'].mean()
        title_utility = "Electricity"
        ax.legend(loc='upper right', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=10)

    else:  # Gas
        bars_g = ax.bar(x, df_filtered['gas_cost'], 0.65,
                        color=GAS_COLOR, alpha=0.85, label='Gas Cost',
                        edgecolor='#0a0e1a', linewidth=0.5, zorder=3)

        for i, cost in enumerate(df_filtered['gas_cost']):
            if cost > 0:
                ax.text(i, cost + 2, f'${cost:.0f}',
                        ha='center', va='bottom', fontsize=8, color=GAS_LIGHT, fontweight='bold')

        avg_cost = df_filtered['gas_cost'].mean()
        title_utility = "Gas"
        ax.legend(loc='upper right', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=10)

    if not np.isnan(avg_cost) and len(df_filtered) > 0:
        ax.axhline(y=avg_cost, color=ACCENT, linestyle='--', linewidth=1.5, alpha=0.7, zorder=4)
        ax.text(len(df_filtered) - 0.5, avg_cost + 4, f'Avg: ${avg_cost:.0f}/mo',
                color=ACCENT, fontsize=10, fontweight='bold', ha='right')

    ax.set_xticks(x)
    ax.set_xticklabels(df_filtered['month_label'], rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Bill Amount ($)', fontsize=13, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.0f}'))
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax.set_title(f'Monthly Bill Amount ({title_utility}: {start_month} – {end_month})', pad=20)

    fig.tight_layout()
    fig.savefig(output_dir / 'chart_monthly_bill.png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()

# Interactive controls
bill_utility_dd = widgets.Dropdown(
    options=['Both', 'Electricity', 'Gas'],
    value='Both',
    description='Utility:',
    layout=widgets.Layout(width='250px')
)

bill_date_slider = widgets.SelectionRangeSlider(
    options=bill_month_options,
    index=(0, len(bill_month_options)-1),
    description='Date Range:',
    layout=widgets.Layout(width='500px')
)

widgets.interact(plot_monthly_bill, utility=bill_utility_dd, date_range=bill_date_slider)


---
## 4. Total Dollar Amount Paid per Year
Total bill amount paid per calendar year — grouped by electricity, gas, and combined. Use the dropdowns to filter for total spend as of a specific month across multiple years (cumulative or single month).

In [ ]:
# ── Chart 3: Total Dollar Amount Paid per Year (Interactive) ──

MONTH_NAMES = ['All Months', 'January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']
MONTH_MAP = {name: i for i, name in enumerate(MONTH_NAMES)}

def plot_yearly_total(utility, selected_month, filter_mode):
    if selected_month == 'All Months':
        e_sub = elec_monthly.copy()
        g_sub = gas_monthly.copy()
        subtitle = "Full Year (Jan – Dec)"
    else:
        m_num = MONTH_MAP[selected_month]
        if filter_mode == 'Cumulative (Jan to Month)':
            e_sub = elec_monthly[elec_monthly['month'] <= m_num].copy()
            g_sub = gas_monthly[gas_monthly['month'] <= m_num].copy()
            subtitle = f"Cumulative (Jan – {selected_month})"
        else:
            e_sub = elec_monthly[elec_monthly['month'] == m_num].copy()
            g_sub = gas_monthly[gas_monthly['month'] == m_num].copy()
            subtitle = f"Single Month ({selected_month} Only)"

    elec_yt = e_sub.groupby('year')['cost'].sum().reset_index().rename(columns={'cost': 'total_elec'})
    gas_yt  = g_sub.groupby('year')['cost'].sum().reset_index().rename(columns={'cost': 'total_gas'})

    yt = pd.merge(elec_yt, gas_yt, on='year', how='outer').fillna(0)
    yt['total_combined'] = yt['total_elec'] + yt['total_gas']
    yt = yt.sort_values('year').reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(12, 7))
    x = np.arange(len(yt))

    max_val = max(yt['total_combined'].max(), yt['total_elec'].max(), yt['total_gas'].max()) if len(yt) > 0 else 100
    offset = max(max_val * 0.015, 2)

    if utility == 'Both':
        bar_width = 0.25
        bars1 = ax.bar(x - bar_width, yt['total_elec'], bar_width,
                       color=ELEC_COLOR, alpha=0.85, label='Electricity',
                       edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        bars2 = ax.bar(x, yt['total_gas'], bar_width,
                       color=GAS_COLOR, alpha=0.85, label='Gas',
                       edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        bars3 = ax.bar(x + bar_width, yt['total_combined'], bar_width,
                       color=ACCENT, alpha=0.85, label='Combined',
                       edgecolor='#0a0e1a', linewidth=0.5, zorder=3)

        for bars, color in [(bars1, ELEC_LIGHT), (bars2, GAS_LIGHT), (bars3, '#d4a5ff')]:
            for bar in bars:
                h = bar.get_height()
                if h > 0:
                    ax.text(bar.get_x() + bar.get_width()/2, h + offset, f'${h:,.0f}',
                            ha='center', va='bottom', fontsize=9.5, color=color, fontweight='bold')
        title_utility = "Electricity + Gas"
        ax.legend(loc='upper left', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=11)
    elif utility == 'Electricity':
        bar_width = 0.55
        bars1 = ax.bar(x, yt['total_elec'], bar_width,
                       color=ELEC_COLOR, alpha=0.85, label='Electricity',
                       edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        for bar in bars1:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + offset, f'${h:,.0f}',
                        ha='center', va='bottom', fontsize=10, color=ELEC_LIGHT, fontweight='bold')
        title_utility = "Electricity"
        ax.legend(loc='upper left', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=11)
    else:  # Gas
        bar_width = 0.55
        bars2 = ax.bar(x, yt['total_gas'], bar_width,
                       color=GAS_COLOR, alpha=0.85, label='Gas',
                       edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
        for bar in bars2:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + offset, f'${h:,.0f}',
                        ha='center', va='bottom', fontsize=10, color=GAS_LIGHT, fontweight='bold')
        title_utility = "Gas"
        ax.legend(loc='upper left', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=11)

    ax.set_xticks(x)
    ax.set_xticklabels(yt['year'].astype(int), fontsize=13)
    ax.set_ylabel('Total Bill Amount ($)', fontsize=13, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax.set_title(f'Total Bill Amount per Year ({title_utility} – {subtitle})', pad=20)

    fig.tight_layout()
    fig.savefig(output_dir / 'chart_total_bill_by_year.png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()

# Interactive controls
yearly_utility_dd = widgets.Dropdown(
    options=['Both', 'Electricity', 'Gas'],
    value='Both',
    description='Utility:',
    layout=widgets.Layout(width='250px')
)

yearly_month_dd = widgets.Dropdown(
    options=MONTH_NAMES,
    value='All Months',
    description='As of Month:',
    layout=widgets.Layout(width='280px')
)

yearly_mode_dd = widgets.Dropdown(
    options=['Cumulative (Jan to Month)', 'Single Month Only'],
    value='Cumulative (Jan to Month)',
    description='Filter Mode:',
    layout=widgets.Layout(width='320px')
)

widgets.interact(plot_yearly_total, utility=yearly_utility_dd, selected_month=yearly_month_dd, filter_mode=yearly_mode_dd)


---
## 5. Average Monthly Bill by Year
Compare the average monthly spend across calendar years — grouped by electricity, gas, and combined.

In [ ]:
# ── Chart 4: Average Monthly Bill by Year ──

# Compute yearly averages
elec_yearly_avg = elec_monthly.groupby('year')['cost'].mean().reset_index()
elec_yearly_avg.columns = ['year', 'avg_elec']

gas_yearly_avg = gas_monthly.groupby('year')['cost'].mean().reset_index()
gas_yearly_avg.columns = ['year', 'avg_gas']

yearly_avg = pd.merge(elec_yearly_avg, gas_yearly_avg, on='year', how='outer').fillna(0)
yearly_avg['avg_total'] = yearly_avg['avg_elec'] + yearly_avg['avg_gas']
yearly_avg = yearly_avg.sort_values('year').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 7))

x = np.arange(len(yearly_avg))
bar_width = 0.25

bars1 = ax.bar(x - bar_width, yearly_avg['avg_elec'], bar_width,
               color=ELEC_COLOR, alpha=0.85, label='Electricity',
               edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
bars2 = ax.bar(x, yearly_avg['avg_gas'], bar_width,
               color=GAS_COLOR, alpha=0.85, label='Gas',
               edgecolor='#0a0e1a', linewidth=0.5, zorder=3)
bars3 = ax.bar(x + bar_width, yearly_avg['avg_total'], bar_width,
               color=ACCENT, alpha=0.85, label='Combined',
               edgecolor='#0a0e1a', linewidth=0.5, zorder=3)

# Value labels
for bars, color in [(bars1, ELEC_LIGHT), (bars2, GAS_LIGHT), (bars3, '#d4a5ff')]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 2, f'${h:.0f}',
                    ha='center', va='bottom', fontsize=10, color=color, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(yearly_avg['year'].astype(int), fontsize=13)
ax.set_ylabel('Average Monthly Bill ($)', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.0f}'))
ax.set_axisbelow(True)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.set_title('Average Monthly Bill by Year', pad=20)
ax.legend(loc='upper right', facecolor='#1a1d29', edgecolor='#2d3250', fontsize=11)

fig.tight_layout()
fig.savefig(output_dir / 'chart_avg_monthly_bill_by_year.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('📊 Saved: chart_avg_monthly_bill_by_year.png')


---
## Summary Statistics

In [ ]:
# ── Summary Stats ──

# Compute yearly totals for summary breakdown
elec_yt_summary = elec_monthly.groupby('year')['cost'].sum().reset_index().rename(columns={'cost': 'total_elec'})
gas_yt_summary  = gas_monthly.groupby('year')['cost'].sum().reset_index().rename(columns={'cost': 'total_gas'})
yearly_total    = pd.merge(elec_yt_summary, gas_yt_summary, on='year', how='outer').fillna(0)
yearly_total['total_combined'] = yearly_total['total_elec'] + yearly_total['total_gas']
yearly_total    = yearly_total.sort_values('year').reset_index(drop=True)

print('='*60)
print('📊 SUMMARY STATISTICS')
print('='*60)

print('\n── Electricity ──')
print(f'  Total usage:           {elec_monthly["usage"].sum():,.0f} kWh')
print(f'  Total cost:            ${elec_monthly["cost"].sum():,.2f}')
print(f'  Avg monthly usage:     {elec_monthly["usage"].mean():,.0f} kWh')
print(f'  Avg daily usage:       {elec_monthly["usage"].sum() / elec_monthly["days"].sum():,.2f} kWh/day')
print(f'  Avg monthly bill:      ${elec_monthly["cost"].mean():,.2f}')
print(f'  Peak month usage:      {elec_monthly.loc[elec_monthly["usage"].idxmax(), "month_label"]} ({elec_monthly["usage"].max():,.0f} kWh)')
print(f'  Lowest month usage:    {elec_monthly.loc[elec_monthly["usage"].idxmin(), "month_label"]} ({elec_monthly["usage"].min():,.0f} kWh)')

print('\n── Gas ──')
print(f'  Total usage:           {gas_monthly["usage"].sum():,.0f} MJ')
print(f'  Total cost:            ${gas_monthly["cost"].sum():,.2f}')
print(f'  Avg monthly usage:     {gas_monthly["usage"].mean():,.0f} MJ')
print(f'  Avg daily usage:       {gas_monthly["usage"].sum() / gas_monthly["days"].sum():,.2f} MJ/day')
print(f'  Avg monthly bill:      ${gas_monthly["cost"].mean():,.2f}')
print(f'  Peak month usage:      {gas_monthly.loc[gas_monthly["usage"].idxmax(), "month_label"]} ({gas_monthly["usage"].max():,.0f} MJ)')
print(f'  Lowest month usage:    {gas_monthly.loc[gas_monthly["usage"].idxmin(), "month_label"]} ({gas_monthly["usage"].min():,.0f} MJ)')

print('\n── Yearly Totals Breakdown ──')
for _, row in yearly_total.iterrows():
    yr = int(row["year"])
    print(f'  {yr}: Electricity: ${row["total_elec"]:,.2f} | Gas: ${row["total_gas"]:,.2f} | Combined: ${row["total_combined"]:,.2f}')

print('\n── Combined ──')
total_all = elec_monthly['cost'].sum() + gas_monthly['cost'].sum()
months_covered = max(len(elec_monthly), len(gas_monthly))
print(f'  Total spend:           ${total_all:,.2f}')
print(f'  Months covered:        {months_covered}')
print(f'  Avg combined monthly:  ${total_all / months_covered:,.2f}')
print('='*60)
